# rustures 사용 튜토리얼

이 노트북은 시계열의 **변화점(change point)** 을 `rustures`로 찾는 전체 흐름을 보여줍니다.

완료하면 다음을 할 수 있습니다.

- `CostL2`로 한 구간이 얼마나 균일한지 계산하기
- 변화점 개수를 아는 경우 `Dynp` 사용하기
- 변화점 개수를 모르고 penalty로 결정할 때 `Pelt` 사용하기
- 평균뿐 아니라 분포 변화도 찾도록 `KernelCPD` 사용하기
- 여러 feature를 가진 2차원 신호 처리하기
- Python으로 Bernoulli custom cost를 작성해 `Dynp`와 `Pelt`에 연결하기

`rustures`의 breakpoint 목록은 항상 마지막 원소로 `n_samples`를 포함합니다. 예를 들어 `[60, 120, 180]`은 `[0, 60)`, `[60, 120)`, `[120, 180)`의 세 구간을 뜻하며 실제 변화점은 60과 120입니다.

## 0. 설치

PyPI에서 `rustures`와 이 노트북의 plotting 의존성을 설치합니다. 현재 개발 환경에 이미 설치되어 있다면 이 셀은 건너뛰십시오.

```bash
python -m pip install rustures matplotlib
```

노트북 안에서 설치하려면 다음 줄의 주석을 해제하십시오.

In [ ]:
# %pip install rustures matplotlib

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
import rustures

np.set_printoptions(precision=3, suppress=True)
print("rustures version:", rustures.__version__)

## 1. 예제 신호 만들기

아래 신호는 180개 관측치로 구성됩니다. 평균이 약 0인 구간, 약 5인 구간, 약 -3인 구간을 차례로 이어 붙였으므로 정답 변화점은 60과 120입니다. 실제 데이터처럼 보이도록 작은 Gaussian 잡음을 더합니다.

변화점 탐지는 관측치 하나가 튀는 위치를 찾는 작업이 아닙니다. **한 가지 모델로 잘 설명되던 구간의 통계적 성질이 바뀌는 경계**를 찾는 작업입니다. 여기서는 각 구간을 하나의 상수 평균으로 설명하는 모델을 사용합니다.

In [ ]:
rng = np.random.default_rng(42)
signal = np.r_[
    rng.normal(0.0, 0.25, 60),
    rng.normal(5.0, 0.25, 60),
    rng.normal(-3.0, 0.25, 60),
].astype(np.float64)
true_bkps = [60, 120, len(signal)]

print("shape:", signal.shape)
print("true breakpoints:", true_bkps)

In [ ]:
def plot_segmentation(values, bkps, title, true_bkps=None):
    """1D/2D 신호와 추정 변화점을 그리는 작은 보조 함수입니다."""
    values = np.asarray(values)
    if values.ndim == 1:
        values = values[:, None]

    fig, axes = plt.subplots(
        values.shape[1], 1,
        figsize=(11, 2.8 * values.shape[1]),
        sharex=True,
        squeeze=False,
    )
    for feature, ax in enumerate(axes[:, 0]):
        ax.plot(values[:, feature], color="#303030", linewidth=1)
        if true_bkps is not None:
            for bkp in true_bkps[:-1]:
                ax.axvline(bkp, color="#2ca02c", linestyle=":", linewidth=2, label="true")
        for bkp in bkps[:-1]:
            ax.axvline(bkp, color="#d62728", linestyle="--", linewidth=2, label="estimated")
        ax.set_ylabel(f"feature {feature}")
        handles, labels = ax.get_legend_handles_labels()
        unique = dict(zip(labels, handles))
        if unique:
            ax.legend(unique.values(), unique.keys(), loc="upper right")
    axes[-1, 0].set_xlabel("sample index")
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

plot_segmentation(signal, [], "Synthetic piecewise-constant signal", true_bkps)

## 2. CostL2: 구간 하나가 한 평균으로 설명되는 정도

`CostL2.error(start, end)`는 구간 `[start, end)` 안에서 각 관측치와 그 구간 평균 사이의 제곱 오차를 모두 더합니다.

$$
C(s,e)=\sum_{t=s}^{e-1}(x_t-\bar{x}_{s:e})^2
$$

한 구간 안에 서로 다른 평균을 가진 데이터가 섞이면 비용이 커집니다. 경계를 잘 나누면 각 구간이 더 균일해져 전체 비용이 작아집니다. 변화점 탐지는 가능한 분할 중에서 다음 합이 작은 분할을 찾는 문제로 볼 수 있습니다.

$$
C(0,b_1)+C(b_1,b_2)+\cdots+C(b_K,N)
$$

`fit`은 빠른 구간 질의를 위한 prefix 통계를 만듭니다. 원본 배열을 나중에 바꾸어도 이미 fit된 비용은 바뀌지 않습니다.

In [ ]:
l2 = rustures.CostL2().fit(signal)

one_regime = l2.error(0, 60)
mixed_regimes = l2.error(0, 120)
manual_total = sum(l2.error(start, end) for start, end in zip([0] + true_bkps[:-1], true_bkps))
library_total = l2.sum_of_costs(true_bkps)

print(f"cost of [0, 60), one regime : {one_regime:.3f}")
print(f"cost of [0, 120), mixed      : {mixed_regimes:.3f}")
print(f"manual total cost            : {manual_total:.3f}")
print(f"sum_of_costs result           : {library_total:.3f}")
assert np.isclose(manual_total, library_total)

## 3. Dynp: 변화점 개수를 알고 있을 때

`Dynp`는 변화점 수 `n_bkps`를 고정하고 전체 구간 비용이 최소인 분할을 동적 계획법으로 찾습니다. 여기서는 실제 변화점이 두 개라는 정보를 넘깁니다.

- `model="l2"`: 각 구간을 상수 평균으로 설명합니다.
- `min_size=3`: 각 구간은 최소 3개 샘플을 포함해야 합니다.
- `jump=1`: 모든 샘플 위치를 경계 후보로 봅니다. `jump`를 키우면 후보 수가 줄어 빨라지지만 근사 결과가 될 수 있습니다.
- `n_bkps=2`: 마지막 `n_samples` 표시는 세지 않고 실제 변화점 두 개를 요구합니다.

동적 계획법은 각 prefix에 대해 '정확히 몇 개 구간으로 나눴을 때의 최솟값'을 저장합니다. 마지막 구간의 시작점만 모두 시도하면, 그 앞부분은 이미 저장된 최적해를 재사용할 수 있습니다. 따라서 가능한 분할 조합을 무식하게 전부 나열하지 않고도 고정된 변화점 수에 대한 전역 최적해를 얻습니다.

In [ ]:
dynp = rustures.Dynp(model="l2", min_size=3, jump=1)
dynp_bkps = dynp.fit_predict(signal, n_bkps=2)

print("Dynp breakpoints:", dynp_bkps)
plot_segmentation(signal, dynp_bkps, "Dynp (L2, fixed K=2)", true_bkps)

## 4. Pelt: 변화점 개수를 모를 때

실제 분석에서는 변화점 수를 미리 모르는 경우가 많습니다. `Pelt`는 구간 비용 합에 변화점 하나당 penalty를 더한 목적함수를 최소화합니다.

$$
\text{objective}=\sum_{j}C(b_{j-1},b_j)+\beta\times(\text{number of changes})
$$

penalty $\beta$가 작으면 구간을 많이 나누고, 크면 단순한 분할을 선호합니다. 데이터 크기와 잡음 수준에 따라 적절한 값이 달라지므로 하나의 보편적인 정답은 없습니다. 여러 penalty를 시도해 변화점 수와 위치가 얼마나 안정적인지 확인하는 것이 좋습니다.

L2 모델에서는 PELT가 앞으로 최적해가 될 수 없는 후보를 안전하게 가지치기합니다. 그래서 결과의 정확성을 유지하면서 실제 계산량을 크게 줄일 수 있습니다.

In [ ]:
for pen in [1.0, 8.0, 1000.0]:
    bkps = rustures.Pelt(model="l2", min_size=3, jump=1).fit_predict(signal, pen=pen)
    print(f"pen={pen:7.1f} -> {bkps} (changes={len(bkps) - 1})")

pelt = rustures.Pelt(model="l2", min_size=3, jump=1).fit(signal)
pelt_bkps = pelt.predict(pen=8.0)
plot_segmentation(signal, pelt_bkps, "Pelt (L2, penalty=8)", true_bkps)

## 5. KernelCPD: 평균만으로 드러나지 않는 변화까지

L2 cost는 평균 변화에 특히 잘 맞습니다. 반면 `KernelCPD`는 원래 관측치를 kernel이 만드는 특징 공간으로 옮겨 그 공간에서 구간의 균일성을 평가합니다. RBF kernel은 평균뿐 아니라 분산이나 분포 모양이 달라지는 변화에도 반응할 수 있습니다.

아래에서는 같은 고정 변화점 수 문제를 `kernel="rbf"`로 풉니다. `backend="fused"`는 큰 Gram 행렬과 전체 구간 비용표를 미리 저장하지 않고 kernel 계산과 DP를 결합한 기본 고성능 경로입니다.

지원 kernel은 `linear`, `rbf`, `cosine`입니다. 어떤 kernel이 적절한지는 무엇을 '같은 구간'으로 볼 것인지에 따라 달라집니다.

In [ ]:
kernel_cpd = rustures.KernelCPD(
    kernel="rbf",
    min_size=3,
    jump=1,
    backend="fused",
)
kernel_bkps = kernel_cpd.fit_predict(signal, n_bkps=2)

print("KernelCPD breakpoints:", kernel_bkps)
print("resolved gamma:", kernel_cpd.gamma)
print("backend:", kernel_cpd.backend)
plot_segmentation(signal, kernel_bkps, "KernelCPD (RBF, fixed K=2)", true_bkps)

## 6. 여러 feature가 있는 신호

입력 shape가 `(n_samples, n_features)`이면 같은 시간의 여러 feature를 한 관측 벡터로 처리합니다. L2 cost는 feature별 구간 평균에서 떨어진 제곱 거리를 모두 합합니다. 즉 feature를 독립적으로 따로 분할하는 것이 아니라, **모든 feature가 공유하는 하나의 변화점 목록**을 찾습니다.

feature의 단위가 크게 다르면 값의 범위가 큰 feature가 비용을 지배할 수 있습니다. 실제 데이터에서는 분석 목적에 맞게 표준화할지 먼저 판단하십시오.

In [ ]:
rng_multi = np.random.default_rng(123)
multi_signal = np.column_stack([
    signal,
    np.r_[
        rng_multi.normal(2.0, 0.35, 60),
        rng_multi.normal(-2.0, 0.35, 60),
        rng_multi.normal(4.0, 0.35, 60),
    ],
]).astype(np.float64)

multi_bkps = rustures.Dynp(model="l2", min_size=3, jump=1).fit_predict(
    multi_signal, n_bkps=2
)
print("input shape:", multi_signal.shape)
print("multivariate breakpoints:", multi_bkps)
plot_segmentation(multi_signal, multi_bkps, "Multivariate Dynp (shared breakpoints)", true_bkps)

## 7. Python custom cost: Bernoulli 확률 변화 찾기

관측치가 0 또는 1이라면 Gaussian 평균 오차보다 Bernoulli 확률 모델이 자연스럽습니다. 한 구간의 성공 확률을 그 구간의 표본 평균으로 추정하고, 그 모델 아래의 음의 로그 가능도(negative log-likelihood)를 구간 비용으로 사용하겠습니다.

custom cost 객체의 계약은 다음과 같습니다.

- 양의 정수 속성 `min_size`
- `fit(signal)`: 신호를 받아 필요한 통계를 준비
- `error(start, end) -> float`: 구간 하나의 유한한 비용 반환
- 선택적 `error_many(starts, ends) -> 1D float64 ndarray`: 여러 구간을 한 번에 계산

아래 구현은 prefix sum을 저장하므로 각 구간에서 1의 개수를 빠르게 구할 수 있습니다. 확률이 정확히 0 또는 1인 구간은 수치적으로 `log(0)`을 계산하지 않고 비용 0으로 처리합니다.

In [ ]:
class BatchBernoulliCost:
    min_size = 2

    def fit(self, signal):
        values = np.asarray(signal, dtype=np.float64)
        if values.ndim == 1:
            values = values[:, None]
        if not np.all((values == 0.0) | (values == 1.0)):
            raise ValueError("Bernoulli observations must be 0 or 1")

        zeros = np.zeros((1, values.shape[1]), dtype=np.float64)
        self.ones_prefix = np.vstack([zeros, np.cumsum(values, axis=0)])
        return self

    @staticmethod
    def _costs(ones, trials):
        mixed = (ones > 0.0) & (ones < trials)
        costs = np.zeros_like(ones, dtype=np.float64)
        probabilities = ones[mixed] / trials[mixed]
        costs[mixed] = (
            -ones[mixed] * np.log(probabilities)
            - (trials[mixed] - ones[mixed]) * np.log1p(-probabilities)
        )
        return costs

    def error(self, start, end):
        ones = self.ones_prefix[end] - self.ones_prefix[start]
        trials = np.full_like(ones, end - start, dtype=np.float64)
        return float(self._costs(ones, trials).sum())

    def error_many(self, starts, ends):
        starts = np.asarray(starts)
        ends = np.asarray(ends)
        ones = self.ones_prefix[ends] - self.ones_prefix[starts]
        lengths = (ends - starts)[:, None].astype(np.float64)
        trials = np.broadcast_to(lengths, ones.shape)
        return self._costs(ones, trials).sum(axis=1).astype(np.float64, copy=False)

In [ ]:
rng_binary = np.random.default_rng(7)
binary_signal = np.r_[
    rng_binary.binomial(1, 0.08, 60),
    rng_binary.binomial(1, 0.92, 60),
    rng_binary.binomial(1, 0.08, 60),
].astype(np.float64)
binary_true_bkps = [60, 120, len(binary_signal)]

bernoulli_dynp = rustures.Dynp(
    custom_cost=BatchBernoulliCost(), min_size=5, jump=1
)
binary_dynp_bkps = bernoulli_dynp.fit_predict(binary_signal, n_bkps=2)

bernoulli_pelt = rustures.Pelt(
    custom_cost=BatchBernoulliCost(), min_size=5, jump=1
)
binary_pelt_bkps = bernoulli_pelt.fit_predict(binary_signal, pen=8.0)

print("Dynp + custom Bernoulli:", binary_dynp_bkps)
print("Pelt + custom Bernoulli:", binary_pelt_bkps)
print("uses custom cost       :", bernoulli_dynp.uses_custom_cost)
print("uses batch callback    :", bernoulli_dynp.uses_batch_callback)
plot_segmentation(
    binary_signal, binary_dynp_bkps,
    "Dynp with a Python Bernoulli cost", binary_true_bkps
)

잡음이 있는 유한 표본에서는 추정 변화점이 생성에 사용한 경계와 몇 칸 다를 수 있습니다. 이는 곧바로 구현 오류를 뜻하지 않습니다. 실제로 관측된 0/1 배열에 대해 가장 작은 목적함수를 만드는 경계가 모집단의 확률을 바꾼 위치와 조금 다를 수 있기 때문입니다.

`error_many`는 알고리즘의 점근적 시간복잡도를 바꾸지는 않지만 Rust와 Python 사이의 호출 횟수를 크게 줄입니다. 이를 생략해도 `error` fallback으로 같은 의미의 계산을 할 수 있지만, 큰 입력에서는 Python callback 오버헤드가 커질 수 있습니다. custom cost를 쓰는 동안에는 Python GIL도 유지됩니다. 최고 성능이 필요하다면 자주 쓰는 cost를 Rust native cost로 추가하는 편이 좋습니다.

custom cost를 사용하는 Pelt는 임의 비용에서 pruning 가정을 함부로 적용하지 않기 위해 exact unpruned optimal partitioning으로 동작합니다. 결과의 정확성을 우선하지만 native L2 Pelt보다 느릴 수 있습니다.

## 8. Python 스레드에서 여러 탐지 작업 실행하기

Rustures의 내장 detector는 native Rust 탐색을 실행하는 동안 Python의 전역 인터프리터 잠금(GIL)을 해제합니다. 따라서 서로 독립적인 CPU 연산 탐지 작업을 `ThreadPoolExecutor`에 넣으면 Python bytecode를 번갈아 실행하는 데 그치지 않고 실제로 동시에 진행할 수 있습니다.

가장 안전한 사용법은 아래처럼 작업마다 detector 인스턴스를 하나씩 만드는 것입니다. fit이 끝난 detector의 읽기 전용 `predict()`도 동시에 호출할 수 있지만, 같은 인스턴스에서 상태를 변경하는 `fit()` 또는 `fit_predict()`를 동시에 호출하면 안 됩니다. Python custom cost는 예외입니다. `error`와 `error_many` callback을 실행할 때 GIL이 필요하므로 내장 cost만큼의 스레드 병렬화 이득을 얻지 못합니다.

실제 가속률은 물리 코어 수, CPU 주파수, 메모리 대역폭, 신호 크기와 시스템 부하에 따라 달라집니다. 작은 작업은 스레드 스케줄링 비용 때문에 오히려 느릴 수 있으므로, 계산량이 충분한 여러 탐지 작업이 있을 때 사용하는 것이 좋습니다.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import os
from time import perf_counter

thread_signals = [
    rustures.pw_constant(
        n_samples=2_500, n_features=3, n_bkps=10,
        noise_std=0.7, seed=100 + index,
    )[0]
    for index in range(8)
]

def detect_one(values):
    # 작업마다 새 detector를 만들어 변경 가능한 fitted state를 공유하지 않습니다.
    return rustures.KernelCPD(
        kernel="linear", backend="fused", min_size=10, jump=1
    ).fit_predict(values, n_bkps=10)

started = perf_counter()
sequential_results = [detect_one(values) for values in thread_signals]
sequential_seconds = perf_counter() - started

workers = min(4, os.cpu_count() or 1)
started = perf_counter()
with ThreadPoolExecutor(max_workers=workers) as pool:
    threaded_results = list(pool.map(detect_one, thread_signals))
threaded_seconds = perf_counter() - started

assert threaded_results == sequential_results
print(f"workers: {workers}")
print(f"순차 실행: {sequential_seconds:.3f}초")
print(f"스레드 실행: {threaded_seconds:.3f}초")
print(f"가속률: {sequential_seconds / threaded_seconds:.2f}배")

## 9. 실전에서의 선택 기준

| 상황 | 우선 시도할 API | 결정해야 할 값 |
|---|---|---|
| 변화점 수를 알고 있고 평균 변화가 핵심 | `Dynp(model="l2")` | `n_bkps` |
| 변화점 수를 모르고 평균 변화가 핵심 | `Pelt(model="l2")` | `pen` |
| 분산·분포 모양 변화도 관심 대상 | `KernelCPD(kernel="rbf")` | `n_bkps` 또는 `pen` |
| 문제 고유의 통계 모델이 필요 | `Dynp/Pelt(custom_cost=...)` | cost 정의와 `n_bkps`/`pen` |

공통 점검 사항은 다음과 같습니다.

1. 입력은 유한한 `float64` 1D 또는 2D NumPy 배열로 준비합니다.
2. `min_size`는 실제로 가능한 가장 짧은 구간보다 작지 않게 정합니다.
3. 정확한 후보 위치가 중요하면 `jump=1`을 사용합니다.
4. 마지막 breakpoint는 실제 변화점이 아니라 신호 끝 `n_samples`임을 기억합니다.
5. 하나의 parameter 결과만 믿기보다 `n_bkps`, penalty, kernel을 바꾸어 결과 안정성을 확인합니다.
6. 탐지 결과는 도메인 지식과 원 신호 시각화로 반드시 검토합니다.

In [ ]:
summary = {
    "Dynp / L2": dynp_bkps,
    "Pelt / L2": pelt_bkps,
    "KernelCPD / RBF": kernel_bkps,
    "Dynp / multivariate L2": multi_bkps,
    "Dynp / custom Bernoulli": binary_dynp_bkps,
    "Pelt / custom Bernoulli": binary_pelt_bkps,
}
for name, bkps in summary.items():
    print(f"{name:27s} -> {bkps}")